# BW \#101 LA Fires
This week, we'll examine the LA fires from the perspective of data using GeoPandas, an extension to Pandas that provides geospatial functionality.
https://geopandas.org/en/stable/index.html?ref=bambooweekly.com 

## Data and six questions
The data comes from the NASA. They, along with the US Forest Service, run FIRMS (https://firms.modaps.eosdis.nasa.gov/usfs/), which provides information about wildfires from its EOS (Earth Observing System) network of satellites. EOS, as the name indicates, looks back at Earth, rather than out into space. Using a variety of sensors, we can learn where fires are taking place, and how hot they are burning. Moreover, the data is frequently updated, giving us information about the current California fires, not just historical data.

The data files are available at https://firms.modaps.eosdis.nasa.gov/usfs/active_fire/
We're going to use the 7-day VIIRS data from NOAA-20. You can download that from the above page, or from this link:

https://firms.modaps.eosdis.nasa.gov/data/active_fire/noaa-20-viirs-c2/csv/J1_VIIRS_C2_USA_contiguous_and_Hawaii_7d.csv

This is a CSV file containing much of the data we want. However, we'll also be using some data about Los Angeles and the surrounding counties. We will get that from the TIGER 2024 data, which includes everything that we need to work with counties:

https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip

Because this is Census data, they don't use state names. Rather, they use "STATEFP" codes, which you can translate from here:

https://www2.census.gov/geo/docs/reference/codes2020/national_state2020.txt

## Challenges
The learning goals involve working with GeoPandas, including joining and plotting. But we'll also do some work with dates and times, non-geo joins, grouping, and pivot tables.
- Create a Pandas data frame from the VIRRS / NOAA-20 data that NASA provides. Include a date column, of dtype datetime, based on the acq_date and acq_time columns. The latter is in HHMM format, reflecting the time (GMT) at which the data was collected. Remove acq_date, acq_time, and satellite when you're done.
- Create a GeoDataFrame based on the data in the regular Pandas data frame you created. Use the latitude and longitude columns to create the special geometry column. Use the EPSG:4326 coordinate reference system (CRS).


In [9]:
#!pip install numpy pandas geopandas scipy matplotlib seaborn scikit-learn tensorflow pytorch keras pyspark scrapy beautifulsoup4 nltk spacy statsmodels


In [10]:
import pandas as pd
import geopandas

First we read the csv file

In [11]:
filename = r"C:\Users\npigeon1\Pandas-Bamboo-Weekly-1\BW #101 LA Fires\J1_VIIRS_C2_USA_contiguous_and_Hawaii_7d.csv"
df = pd.read_csv(filename)
df.head()

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,confidence,version,bright_ti5,frp,daynight
0,41.43517,-75.59569,323.60,0.39,0.44,2025-01-13,629,N20,nominal,2.0NRT,267.78,2.14,N
1,23.08642,-81.27223,307.76,0.48,0.48,2025-01-13,635,N20,nominal,2.0NRT,291.15,1.20,N
2,41.43834,-75.59781,317.54,0.39,0.44,2025-01-13,629,N20,nominal,2.0NRT,266.50,2.12,N
3,22.88769,-82.74165,307.52,0.59,0.53,2025-01-13,635,N20,nominal,2.0NRT,290.14,1.86,N
4,22.65450,-81.23954,303.15,0.47,0.48,2025-01-13,635,N20,nominal,2.0NRT,289.40,0.79,N


Next we combine the columns acq_date and acq_time into a date column of dtype datetime. We can do that if we have a string column in a format that pd.to_datetime recognizes – or we can pass a format string (as specified in such places as https://www.strfti.me/) to give it a further hint. The problem is that the acq_time column is seen as integers by read_csv. Moreover, it's supposed to be a four-digit time in HHMM format, but sometimes it's just HMM.

In [12]:
df.dtypes

latitude      float64
longitude     float64
bright_ti4    float64
scan          float64
track         float64
acq_date       object
acq_time        int64
satellite      object
confidence     object
version        object
bright_ti5    float64
frp           float64
daynight       object
dtype: object

However, we can't use acq_time, because it's an integer column. Instead, we'll use astype to turn it into a string column. We'll then use str.zfill to pad our string with leading zeroes, ensuring that we end up with four characters total.

The result will be a string in the format of YYYY-MM-DD HHMM. We can tell pd.to_datetime to use this format by passing the format keyword argument '%Y-%m-%d %H%M'. In other words, we end up with:

In [13]:
df['date'] = pd.to_datetime(
    df['acq_date'] + ' ' + df['acq_time'].astype(str).str.zfill(4),
    format='%Y-%m-%d %H%M'
)

Next we remove acq_date, acq_time, and satellite when you're done.

In [14]:
df.drop(columns = ['acq_date', 'acq_time', 'satellite'])

,latitude,longitude,bright_ti4,scan,track,confidence,version,bright_ti5,frp,daynight,date
0,41.43517,-75.59569,323.60,0.39,0.44,nominal,2.0NRT,267.78,2.14,N,2025-01-13 06:29:00
1,23.08642,-81.27223,307.76,0.48,0.48,nominal,2.0NRT,291.15,1.20,N,2025-01-13 06:35:00
2,41.43834,-75.59781,317.54,0.39,0.44,nominal,2.0NRT,266.50,2.12,N,2025-01-13 06:29:00
3,22.88769,-82.74165,307.52,0.59,0.53,nominal,2.0NRT,290.14,1.86,N,2025-01-13 06:35:00
4,22.65450,-81.23954,303.15,0.47,0.48,nominal,2.0NRT,289.40,0.79,N,2025-01-13 06:35:00
...,...,...,...,...,...,...,...,...,...,...,...
8683,29.17250,-107.12057,345.99,0.49,0.48,nominal,2.0NRT,293.28,12.27,D,2025-01-20 20:42:00
8684,29.56222,-106.02356,340.78,0.57,0.52,nominal,2.0NRT,303.01,5.36,D,2025-01-20 20:42:00
8685,29.45070,-110.31012,327.62,0.49,0.41,low,2.0NRT,299.11,1.84,D,2025-01-20 20:42:00
8686,30.19827,-107.68657,339.96,0.46,0.47,nominal,2.0NRT,296.00,10.48,D,2025-01-20 20:42:00


What we'll do is use assign to create a new date column. Its contents will be the result of invoking pd.to_datetime on a combination of acq_date and acq_time. We use a lambda expression here, because pd.to_datetime isn't a method, and thus needs to be invoked in another context.

In [15]:
df = (
    pd
    .read_csv(filename)
    .assign(date = lambda df_: pd.to_datetime(
        df_['acq_date'] + ' ' + df_['acq_time'].astype(str).str.zfill(4),
        format='%Y-%m-%d %H%M'))
    .drop(columns=['acq_date', 'acq_time', 'satellite'])
)

The result is a data frame with 7,895 rows and 11 columns.



## Create a GeoDataFrame based on the data in the regular Pandas data frame you created. Use the latitude and longitude columns to create the special geometry column. Use the EPSG:4326 coordinate reference system (CRS).

GeoPandas defines a subclass of DataFrame known as a GeoDataFrame. The big difference between the two is that everyGeoDataFrame has a special geometry column, which we can use to perform special geographic calculations. In all other ways, a GeoDataFrame is the same as a regular data frame.
To get a GeoDataFrame from what we've created in df, we need to invoke geopandas.GeoDataFrame, passing it df. But then we need to tell it how to define the geometry column. In this case, it's pretty simple – df has longitude and latitude columns, and GeoPandas has a special geopandas.points_from_xy function, designed for precisely these occasions.

We invoke the function on df, passing the keyword arguments that tell it what to use for longitude and latitude. We also have to indicate which coordinate reference system (CRS) we want to use; in this case, I asked you to choose EPSG:4326, which is often used in GPS systems (https://epsg.io/4326).

We create gdf, the GeoDataFrame, and it's just like df was before it – but now it has a geometry column, one with POINT objects that represent the location of where the satellite picked up information.


In [16]:
gdf = geopandas.GeoDataFrame(
    df,
    geometry=geopandas.points_from_xy(df['longitude'], df['latitude']),
    crs="EPSG:4326"
)
gdf.head()

,latitude,longitude,bright_ti4,scan,track,confidence,version,bright_ti5,frp,daynight,date,geometry
0,41.43517,-75.59569,323.60,0.39,0.44,nominal,2.0NRT,267.78,2.14,N,2025-01-13 06:29:00,POINT (-75.59569 41.43517)
1,23.08642,-81.27223,307.76,0.48,0.48,nominal,2.0NRT,291.15,1.20,N,2025-01-13 06:35:00,POINT (-81.27223 23.08642)
2,41.43834,-75.59781,317.54,0.39,0.44,nominal,2.0NRT,266.50,2.12,N,2025-01-13 06:29:00,POINT (-75.59781 41.43834)
3,22.88769,-82.74165,307.52,0.59,0.53,nominal,2.0NRT,290.14,1.86,N,2025-01-13 06:35:00,POINT (-82.74165 22.88769)
4,22.65450,-81.23954,303.15,0.47,0.48,nominal,2.0NRT,289.40,0.79,N,2025-01-13 06:35:00,POINT (-81.23954 22.6545)
